# RNN and Natural Language Processing

In this exercise we will try to classify IMDB dataset: Given the text of a review, can you predict if the review was positive or negative?

Before doing this exercise, you might want to become more familiar with LSTMs by considering the example FlightPassengerPredictions.

The data for this exercise can be found here:
https://sid.erda.dk/share_redirect/encok5nw3y

***

Last Update: 2026-11-11

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import pickle
from collections import defaultdict
from torch import nn
import json
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import chain
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## Data

In [4]:
limit_data = 10000  # limit the amount of data for speed, change as you please

def remove_special_symbols(string):
    return ''.join(s for s in string if ord(s)>96 and ord(s)<123 or s == ' ')

with open('train.json') as f:
    train_text, train_labels = json.load(f)
    idxs = np.random.permutation(len(train_text))
    train_text = [remove_special_symbols(train_text[i]) for i in idxs[:limit_data]]
    train_labels = [train_labels[i] for i in idxs[:limit_data]]
    
with open('test.json') as f:
    test_text, test_labels = json.load(f)
    idxs = np.random.permutation(len(test_text))   
    test_text = [remove_special_symbols(test_text[i]) for i in idxs[:limit_data]]
    test_labels = [test_labels[i] for i in idxs[:limit_data]]

Let's have a look at the data... Here is a negative review (label = 0)

In [5]:
print(train_text[0])
print(train_labels[0])

he movie was actually a romantic drama based on three sisters who had desires to become a famous girl group n their endeavors the oldest sister meets a drug dealer and street hustler called atin whom ister goes after because she believes he is the big time who will give her everything she ever thought she wanted out of life hough he could be accused of killing her he really kills only her spirit and will to live after which she becomes a drug addict and ultimately dies from an overdose he story isnt about the street life or the talian mobster who tries to buy tix off then threatens him its about how love can overcome even the worst tragedies in life as portrayed in song and style and the character that was the life in the times for young women trying to be discovered back then
1


And a positive one:

In [6]:
print(train_text[3])
print(train_labels[3])

 am completely shocked that this show had been cancelledty only lasted one year just recently started watching it and  love itts a show that could of gone as far as riends wentt had the humour and was extremely enjoyablebr br t is about  brothers and  sisters living under one roof without their parentsurtoey awrence plays the part of the oldest sibling and takes on the more fatherly rolebr br his should of lasted much more than year as it was fantastic mazing show with all the best actorsbr br 
1


## Embedding

Let's have a look at all the words we have:

In [7]:
all_words = list(chain(*[x.lower().split() for x in train_text]))
print('total number of unique words =', len(set(all_words)))

total number of unique words = 74272


That's a lot of words... of course, we could clean the data even more if we wanted to. But we won't...
(for instance, there are probably many misspelled words)

In [8]:
Counter(all_words)

Counter({'the': 115322,
         'and': 61633,
         'a': 61622,
         'of': 56451,
         'to': 52893,
         'is': 42777,
         'in': 34291,
         'he': 27242,
         'that': 26826,
         'it': 26005,
         'this': 23895,
         'br': 23132,
         'was': 19261,
         'as': 17403,
         'with': 16860,
         'his': 16857,
         'for': 16541,
         'movie': 16502,
         'film': 14799,
         'but': 14016,
         'on': 13241,
         'are': 11606,
         'not': 10950,
         'have': 10826,
         'be': 10440,
         'you': 10352,
         'one': 9524,
         'an': 8870,
         'at': 8819,
         'by': 8486,
         'all': 8454,
         'from': 7742,
         'who': 7725,
         'like': 7535,
         'or': 7509,
         'they': 7275,
         'its': 7271,
         'her': 6965,
         'so': 6844,
         'about': 6694,
         'just': 6557,
         'has': 6543,
         'out': 6442,
         'some': 5671,
        

In [9]:
n_words = 25000   # let's make a model that only understand 25000 words

In [10]:
words, count = np.unique(all_words, return_counts=True)
idxs = np.argsort(count)[-n_words:]
vocab = ['<UNK>'] + list(words[idxs][::-1])
print(vocab[:5], '...', vocab[-5:])

['<UNK>', 'the', 'and', 'a', 'of'] ... ['hostbusters', 'detrimental', 'clockwork', 'judgments', 'accentsbr']


Not very surprisingly, the most commonly used word is _the_. The 25000th most used word is _chimneys_. We have added a special word `<UNK>` which we will used to mark words outside our vocabulary.

We can now turn a sentence into a sequence of integers that correspond to the position in the vocab.

In [11]:
vocab_d = {vocab[i]: i for i in range(len(vocab))}  # for quick look-up
def sentence_to_integer_sequence(s):
    return torch.tensor([vocab_d[x] if x in vocab_d else 0 for x in s.split()], dtype=torch.long)

In [12]:
sentence_to_integer_sequence("i really liked the movie xenopus51")

tensor([140,  53, 408,   1,  18,   0])

#### Representing high dimensional spaces with a simpler (learnable) embedding:

We are now representing words in a "25000"-dimensional space: we have a unique integer for each word we can represent. To reduce this complexity, we instead intend to represent each word as a 50-dimensional real vector. Pytorch to the rescue:

In [13]:
embedding = nn.Embedding(len(vocab), 50)

`nn.Embedding` assigns are random, **but trainable** vector to each word. For instance:

In [14]:
print(embedding(sentence_to_integer_sequence("movie")))

tensor([[ 1.5340,  1.0444, -0.3893,  0.9449, -0.2459, -1.0086,  0.7491, -0.4766,
          0.8534,  0.8723,  0.9367, -0.2347, -0.8318,  0.3207,  0.7303,  0.9125,
         -1.9431, -0.3792,  0.2270,  0.2099, -1.3233, -0.8218,  1.5787, -1.7354,
         -2.1293, -0.0360, -1.3894, -0.2065,  0.1012, -0.1797, -2.0594, -0.1878,
         -0.0485,  1.3553, -0.7001, -0.1760, -2.4927, -2.5244,  0.3134, -1.1999,
         -1.5704, -0.0726,  1.4387,  1.6155,  0.0312,  0.1223, -0.6284, -1.0808,
         -0.9910,  1.3108]], grad_fn=<EmbeddingBackward0>)


The special `<UNK>` word, which signifies unknown we can choose to zero out:

In [15]:
embedding.weight.data[0, :] = 0

Here is an example of how we represent a sentence then:

In [16]:
print(embedding(sentence_to_integer_sequence("i really liked the movie xenopus51")).shape)
print(embedding(sentence_to_integer_sequence("i really liked the movie xenopus51")))

torch.Size([6, 50])
tensor([[ 0.1125, -0.0944,  0.1229, -0.3131, -0.6234, -0.7197, -0.3412,  0.3041,
          1.2551, -0.1318, -1.8874, -0.1352,  0.7404,  0.4011,  1.0146,  1.1244,
         -0.7478, -0.5869,  0.3130,  0.8744, -0.0310,  0.8320,  0.7107,  0.4848,
         -0.7267,  0.4743, -0.5833,  2.0458,  0.1717,  0.6440, -0.2894, -0.8652,
         -0.0139, -1.6954,  0.6310, -0.2967,  0.1174, -0.1088,  0.6212, -0.3491,
         -0.7596,  1.4025,  2.1422,  2.3165,  0.5473,  0.6897, -0.6375, -0.4211,
         -1.2259,  0.3729],
        [-0.1062,  0.4851, -0.2536, -0.7852, -0.2482,  0.1916, -0.2251,  0.5502,
         -1.2374, -0.6828,  0.6581,  0.0924, -0.1237,  1.2461, -0.1206, -0.6800,
          0.2476,  0.5202, -0.4350,  0.3850, -0.6417, -0.6674, -0.8780, -0.5902,
          0.5487,  0.1213, -1.6016, -2.0280, -0.8701, -0.9477,  1.5687,  0.2131,
         -0.5580,  0.7920,  0.0236,  0.2867,  0.2481,  1.0187, -0.2223, -1.9235,
          0.2690, -0.1358, -1.3703, -0.2307, -2.3342,  0.6893

In this way, the sentence bascially becomes a 6x50 pixel image.

You can now use `nn.LSTM` after an embedding to define a neural network for sentenes.

This network will have a _lot_ of parameters. For each word, 50 parameters needs to be trained, and then comes the LSTM on top of that.
This is the reason that _transfer learning_ is so important in natural language processing (NLP).

Perhaps the simplest form of transfer learning is to use a pretrained embedding layer.

In [17]:
with open('glove.6B.50d.pkl', 'rb') as f:
    glove = pickle.load(f)

In [18]:
glove['movie']

array([ 0.30824 ,  0.17223 , -0.23339 ,  0.023105,  0.28522 ,  0.23076 ,
       -0.41048 , -1.0035  , -0.2072  ,  1.4327  , -0.80684 ,  0.68954 ,
       -0.43648 ,  1.1069  ,  1.6107  , -0.31966 ,  0.47744 ,  0.79395 ,
       -0.84374 ,  0.064509,  0.90251 ,  0.78609 ,  0.29699 ,  0.76057 ,
        0.433   , -1.5032  , -1.6423  ,  0.30256 ,  0.30771 , -0.87057 ,
        2.4782  , -0.025852,  0.5013  , -0.38593 , -0.15633 ,  0.45522 ,
        0.04901 , -0.42599 , -0.86402 , -1.3076  , -0.29576 ,  1.209   ,
       -0.3127  , -0.72462 , -0.80801 ,  0.082667,  0.26738 , -0.98177 ,
       -0.32147 ,  0.99823 ])

These pretrained word embeddings will have a good structure to them. For instance:

In [19]:
print('Distance b/w queen and prince =', np.linalg.norm(glove['queen'] - glove['prince']))
print('Distance b/w movie and prince =', np.linalg.norm(glove['movie'] - glove['prince']))

Distance b/w queen and prince = 3.3926491063677284
Distance b/w movie and prince = 6.450784821820618


You can even sometimes get away with doing algebra with these vectors:

In [20]:
queenlike = glove['king'] - glove['man'] + glove['woman']

In [21]:
print('Distance b/w queen and algebraic queen =', np.linalg.norm(glove['queen'] - queenlike))
print('Distance b/w queen and king =', np.linalg.norm(glove['queen'] - glove['king']))

Distance b/w queen and algebraic queen = 2.8391206432941996
Distance b/w queen and king = 3.4777562289742345


We can fill out our embedding layer using these pretrained vectors:

In [22]:
filled = 0
not_found = []
for i, w in enumerate(vocab):
    if w in glove:
        embedding.weight.data[i, :] = torch.tensor(glove[w], dtype=torch.float)
        filled += 1
    else:
        not_found.append(w)
print(f'Of the words in the vocab, {100 * filled / (len(vocab) - 1)} % were updated using Glove vectors')
print('Examples of words not found :', not_found[1:7])

Of the words in the vocab, 73.6 % were updated using Glove vectors
Examples of words not found : ['owever', 'ichael', 'itbr', 'ritish', 'moviebr', 'nglish']


Clearly the words we do not find are due to misspellings. You now have three choices before you continue with the exercise:

(1) Use _hunspell_ or similar to fix misspelled words

(2) Change the vocabulary (`vocab`) to be based on words that are both frequent in the text and have glove vectors

(3) Ignore the issue.

Finally, with Glove vectors we do not need to train the embedding. We can consider it fixed:

In [23]:
embedding.requires_grad_(False)

Embedding(25001, 50)

### Exercise 1

Train a neural network using `nn.LSTM` layers to classify the IMDB reviews.

Ideas:

 - The sentences can be quite long, so you might want to limit them to, say, maximum 100 words
 - `nn.LSTM` can take batched input, but be careful with `batch_first=True/False`.
 - If batched input are used, they normally have to be equal in size. You can put sentences to always be 100 words long, by adding `<UNK>` words to short sentences.
 - Alternatively, `nn.LSTM` can also accept batched, variable-length input using `torch.nn.utils.rnn.pack_sequence`.
 - (a finaly, albeit slow, alternative is to simple run the LSTM on un-batched input)


## Language model

Often in NLP, a lot of unlabelled text is available. We can use this to pretrain the model, before fine-tuning to the task at hand.

In [24]:
with open('unlabelled.json') as f:
    text = json.load(f)
print(len(text))

50000


A simple way to pretrain is to train a _language model_. This is a model that tries to predict the next word in a sentence. For instance, given:

In [25]:
" ".join(text[2].split()[:28])

'This is still a pretty bad and silly simplistic typical slasher but when being compared to the previous sequel "Slumber Party Massacre II" this movie is a step'

can you guess the next word?

In a language model, we consider the above the input, and the output is:

In [26]:
text[2].split()[28]

'up'

The input we encode using the embedding, while the output is a probability map over words.
In other words, the last layer is something like `nn.Linear(..., len(vocab))`.

### Exercise 2 (optional)

Train a language model.

After you have trained the model, try to make to complete sentences.

## Transfer learning
### Exercise 3 (optional)

Discard the last layer of the now trained language model and use it to train on the original IMDB-problem.